In [1]:
import pandas as pd
import requests
import time
import openpyxl 

In [2]:
# Query all hospitals in Lagos State
query = """
[out:json];
area["name"="Lagos"]["boundary"="administrative"]->.searchArea;
node["amenity"="hospital"](area.searchArea);
out body;
"""

response = requests.get(
    "https://overpass-api.de/api/interpreter",
    params={"data": query}
)

if response.status_code != 200:
    print(f"Error {response.status_code} from Overpass API: {response.text}")
    exit(1)

try:
    data = response.json()
except requests.exceptions.JSONDecodeError:
    print(f"Failed to parse JSON response: {response.text}")
    exit(1)

hospitals = data.get("elements", [])
print(f"Found {len(hospitals)} hospitals from OpenStreetMap. Processing addresses...")

results = []

for i, element in enumerate(hospitals, 1):
    tags = element.get("tags", {})
    name = tags.get("name", "Unknown")
    lat = element.get("lat")
    lon = element.get("lon")
    
    # Try to build address from tags
    address_parts = []
    if "addr:housenumber" in tags:
        address_parts.append(tags["addr:housenumber"])
    if "addr:street" in tags:
        address_parts.append(tags["addr:street"])
    if "addr:city" in tags:
        address_parts.append(tags["addr:city"])
        
    full_address = tags.get("addr:full")
    
    if full_address:
        address = full_address
    elif address_parts:
        address = ", ".join(address_parts)
    else:
        address = None

    if not address and lat and lon:
        # If no address tag was found, reverse geocode the coordinates
        print(f"[{i}/{len(hospitals)}] Address missing. Reverse geocoding for: {name}...")
        try:
            # Nominatim Public API Usage Policy strictly limits to max 1 request per second
            time.sleep(1.2)  
            rev_res = requests.get(
                f"https://nominatim.openstreetmap.org/reverse?lat={lat}&lon={lon}&format=json",
                headers={'User-Agent': 'HospitalAddressScraper/1.0 (local)'}
            )
            if rev_res.status_code == 200:
                address = rev_res.json().get("display_name", "Not available")
            else:
                address = "Not available"
        except Exception:
            address = "Not available"
    elif not address:
        address = "Not available"

    results.append({
        "Hospital": name,
        "Address": address,
        "Latitude": lat,
        "Longitude": lon
    })

Found 473 hospitals from OpenStreetMap. Processing addresses...
[1/473] Address missing. Reverse geocoding for: Rojafia Clinic...
[3/473] Address missing. Reverse geocoding for: Eti-Osa...
[5/473] Address missing. Reverse geocoding for: Safeway hospital...
[6/473] Address missing. Reverse geocoding for: Caremax Hospital...
[7/473] Address missing. Reverse geocoding for: Sangotedo Primary Health Care Center...
[9/473] Address missing. Reverse geocoding for: Grace Olambipo Clinic...
[10/473] Address missing. Reverse geocoding for: Isashi Primary Health Care Center...
[11/473] Address missing. Reverse geocoding for: Lagos State Ministry Of Health Belinda Convalscent And Fertility Centre...
[12/473] Address missing. Reverse geocoding for: Pioneer Medical Clinic...
[13/473] Address missing. Reverse geocoding for: Leverage Hospital Specialist Clinic...
[14/473] Address missing. Reverse geocoding for: Tower Of Glory Hospital...
[15/473] Address missing. Reverse geocoding for: Noble Medical Di

In [3]:
df = pd.DataFrame(results)
df.to_excel("hospitals_lagos.xlsx", index=False)
print("Finished! Saved all data to hospitals_lagos.xlsx")

Finished! Saved all data to hospitals_lagos.xlsx


In [10]:
import sys
# Add the folder containing your script so Jupyter can find it
sys.path.append(r"c:\Users\USER\.antigravity\GPS VENV\address_venv")
# Now you can import your brand new function!
from take_input import search_hospitals

# Now you can call the function with your list
hospitals = ["Premier Specialist Medical Centre", "Etta Atlantic Memorial Hospital", "Isalu Hospitals", "Mercy Stripes Specialist Hospital", "Wind of Grace Hospital", "Lifeline Children's Hospital", "Lagoon Specialist Suites", "CarePoint Hospital", "BlueCross Hospital", "Newgate Medical Services", "St. Raphael Divine Mercy Hospital", "Springfield Hospital", "Crystal Specialist Hospital", "Skipper Eye-Q Hospital", "Eye Foundation Hospital", "Optimal Specialist Hospital", "Budo Specialist Hospital"]
results = search_hospitals(hospitals)
display(results)


Searching Nominatim API for 17 hospital(s)...

[1/17] Searching for: Premier Specialist Medical Centre...
  -> No results found for 'Premier Specialist Medical Centre' in Lagos.
[2/17] Searching for: Etta Atlantic Memorial Hospital...
  -> No results found for 'Etta Atlantic Memorial Hospital' in Lagos.
[3/17] Searching for: Isalu Hospitals...
  -> No results found for 'Isalu Hospitals' in Lagos.
[4/17] Searching for: Mercy Stripes Specialist Hospital...
  -> No results found for 'Mercy Stripes Specialist Hospital' in Lagos.
[5/17] Searching for: Wind of Grace Hospital...
  -> No results found for 'Wind of Grace Hospital' in Lagos.
[6/17] Searching for: Lifeline Children's Hospital...
  -> No results found for 'Lifeline Children's Hospital' in Lagos.
[7/17] Searching for: Lagoon Specialist Suites...
  -> No results found for 'Lagoon Specialist Suites' in Lagos.
[8/17] Searching for: CarePoint Hospital...
  -> No results found for 'CarePoint Hospital' in Lagos.
[9/17] Searching for: Bl

,Hospital,Address,Latitude,Longitude
0,Premier Specialist Medical Centre,Not found,N/A,N/A
1,Etta Atlantic Memorial Hospital,Not found,N/A,N/A
2,Isalu Hospitals,Not found,N/A,N/A
3,Mercy Stripes Specialist Hospital,Not found,N/A,N/A
4,Wind of Grace Hospital,Not found,N/A,N/A
5,Lifeline Children's Hospital,Not found,N/A,N/A
6,Lagoon Specialist Suites,Not found,N/A,N/A
7,CarePoint Hospital,Not found,N/A,N/A
8,BlueCross Hospital,Not found,N/A,N/A
9,Newgate Medical Services,Not found,N/A,N/A


In [21]:
from hospitals import hospitals

In [24]:
data = hospitals.split("\n")
print(data)

['Rojafia Clinic', 'Balogun Primary Healthcare Center.', 'Eti-Osa', 'Pink Clinic Nigeria', 'Safeway hospital', 'Caremax Hospital', 'Sangotedo Primary Health Care Center', 'The Tent Wellness Centre', 'Grace Olambipo Clinic', 'Isashi Primary Health Care Center', 'Lagos State Ministry Of Health Belinda Convalscent And Fertility Centre', 'Pioneer Medical Clinic', 'Leverage Hospital Specialist Clinic', 'Tower Of Glory Hospital', 'Noble Medical Digonistic Centre', 'Grace Land Medical Center', 'Pa Clement Ogunsusi', 'Adis Bunm Trado Medical Hospital And Maternity Home 14 Nnaema Uzor Street Off Summit Light Road', 'Ndieze Homeopathic Medical Services 20 Drchiwumba Off Summit Light Road', 'Wisdom Hospital And Maternity', 'Betheland Hospital', 'Ademola Medical Center', 'Hyssop Eye Clinic', 'Ore Ofe Hospital Ijanikin', 'Femola Hospital', 'Ijanikin Primary Health Care Center By Ile Oba Bstop Lagos Badagry Express Road', 'Angelic Care Foundation Maternity', 'Ever Life Hospital', 'Era Primary Health